# Multi-Agent Orchestration

Design and implement production multi-agent systems — supervisor patterns, debate protocols, shared memory, and complex orchestration.

**Topics:** Agent communication patterns, supervisor-worker, debate pattern, shared state management, multi-agent research system

## 1. Multi-Agent Communication Patterns

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
from enum import Enum
import json
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

class MessageType(Enum):
    TASK = 'task'
    RESULT = 'result'
    FEEDBACK = 'feedback'
    BROADCAST = 'broadcast'

@dataclass
class AgentMessage:
    sender: str
    recipient: str  # or 'ALL' for broadcast
    msg_type: MessageType
    content: str
    metadata: dict = field(default_factory=dict)

@dataclass 
class Agent:
    name: str
    role: str
    system_prompt: str
    model: str = 'gpt-4o-mini'
    inbox: list[AgentMessage] = field(default_factory=list)
    
    def respond(self, message: str, context: list[dict] = None) -> str:
        messages = [{'role': 'system', 'content': self.system_prompt}]
        if context:
            messages.extend(context)
        messages.append({'role': 'user', 'content': message})
        response = client.chat.completions.create(model=self.model, messages=messages, max_tokens=512)
        return response.choices[0].message.content

# Communication patterns
patterns = {
    'Sequential': 'Agent A → Agent B → Agent C (pipeline)',
    'Parallel':   'All agents work simultaneously, results merged',
    'Hierarchical': 'Supervisor delegates to workers, aggregates results',
    'Collaborative': 'Agents share context, no strict hierarchy',
    'Competitive': 'Multiple agents propose solutions, best selected',
    'Debate':     'Agents argue for/against, consensus reached',
}

print('Multi-agent communication patterns:')
for pattern, desc in patterns.items():
    print(f'  {pattern:<15}: {desc}')
print()
print('Pattern selection guide:')
print('  Complex task decomposition → Hierarchical (supervisor-worker)')
print('  Multiple independent analyses → Parallel')
print('  Need validation/verification → Debate or Competitive')
print('  Sequential refinement → Sequential pipeline')

## 2. Supervisor-Worker Architecture

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

SUPERVISOR_PROMPT = """You are a supervisor coordinating a team of specialist agents.
Break down the task and assign sub-tasks to the right specialist.

Available specialists:
- researcher: Finds and summarizes relevant information
- analyst: Analyzes data and draws conclusions
- writer: Produces clear, well-structured written output
- critic: Reviews work and identifies issues

Output your delegation plan as JSON: {"assignments": [{"agent": "name", "task": "..."}]}"""

async def run_worker(agent_name: str, role: str, task: str, context: str = '') -> dict:
    """Run a single worker agent on its assigned task."""
    system_prompts = {
        'researcher': 'You are an expert researcher. Find and summarize key information concisely.',
        'analyst':    'You are a data analyst. Analyze inputs and provide data-driven insights.',
        'writer':     'You are a skilled technical writer. Produce clear, structured content.',
        'critic':     'You are a rigorous critic. Identify flaws, gaps, and improvements.',
    }
    
    messages = [{
        'role': 'system', 'content': system_prompts.get(role, 'You are a helpful assistant.')
    }]
    if context:
        messages.append({'role': 'user', 'content': f'Context from other agents:\n{context}'})
        messages.append({'role': 'assistant', 'content': 'Understood, I will incorporate this context.'})
    messages.append({'role': 'user', 'content': task})
    
    response = await async_client.chat.completions.create(
        model='gpt-4o-mini', messages=messages, max_tokens=512
    )
    return {'agent': agent_name, 'result': response.choices[0].message.content}

async def supervisor_pipeline(task: str) -> str:
    """Supervisor delegates subtasks, workers run in parallel, supervisor synthesizes."""
    # Step 1: Supervisor creates delegation plan
    plan_response = client.chat.completions.create(
        model='gpt-4o', messages=[
            {'role': 'system', 'content': SUPERVISOR_PROMPT},
            {'role': 'user', 'content': f'Task: {task}'},
        ],
        response_format={'type': 'json_object'},
    )
    plan = json.loads(plan_response.choices[0].message.content)
    
    # Step 2: Run workers in parallel
    worker_tasks = [
        run_worker(a['agent'], a['agent'], a['task'])
        for a in plan['assignments']
    ]
    results = await asyncio.gather(*worker_tasks)
    
    # Step 3: Supervisor synthesizes
    context = '\n\n'.join(f"{r['agent']}: {r['result']}" for r in results)
    final = client.chat.completions.create(
        model='gpt-4o', messages=[
            {'role': 'system', 'content': 'Synthesize the worker outputs into a final answer.'},
            {'role': 'user', 'content': f'Task: {task}\n\nWorker outputs:\n{context}'},
        ],
    )
    return final.choices[0].message.content

print('Supervisor-worker pipeline:')
print('  1. Supervisor analyzes task → creates delegation plan (JSON)')
print('  2. Workers run in parallel with asyncio.gather()')
print('  3. Supervisor reads all results → synthesizes final answer')
print()
print('Latency vs sequential:')
n_workers = 4
worker_time = 2.0
print(f'  Sequential: {n_workers} × {worker_time}s = {n_workers * worker_time}s')
print(f'  Parallel:   max({worker_time}s) = {worker_time}s ({n_workers}x speedup!)')

## 3. Debate Pattern — Agents Argue for Best Answer

In [ ]:
DEBATER_PROMPT = """You are Agent {name}, a {stance} advocate.
Your role: argue {stance} for the proposed solution.
Be specific, cite evidence, and challenge the opposing view.
Keep responses concise (2-3 paragraphs)."""

JUDGE_PROMPT = """You are an impartial judge evaluating a debate.
Read the arguments from both sides and make a final decision.
Be objective and explain your reasoning.
Output JSON: {"winner": "for/against/tie", "reasoning": "...", "final_recommendation": "..."}"""

def run_debate(
    proposal: str,
    n_rounds: int = 2,
    judge_model: str = 'gpt-4o',
) -> dict:
    """Multi-round debate to stress-test proposals before implementation."""
    debate_history = [f'Proposal to debate: {proposal}']
    
    for round_num in range(n_rounds):
        # Agent FOR argues in favor
        for_response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': DEBATER_PROMPT.format(name='Alpha', stance='for')},
                {'role': 'user', 'content': '\n'.join(debate_history)},
            ],
        ).choices[0].message.content
        debate_history.append(f'[Round {round_num+1} FOR]: {for_response}')
        
        # Agent AGAINST argues against
        against_response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': DEBATER_PROMPT.format(name='Beta', stance='against')},
                {'role': 'user', 'content': '\n'.join(debate_history)},
            ],
        ).choices[0].message.content
        debate_history.append(f'[Round {round_num+1} AGAINST]: {against_response}')
    
    # Judge makes final decision
    verdict = client.chat.completions.create(
        model=judge_model,
        messages=[
            {'role': 'system', 'content': JUDGE_PROMPT},
            {'role': 'user', 'content': '\n\n'.join(debate_history)},
        ],
        response_format={'type': 'json_object'},
    )
    return json.loads(verdict.choices[0].message.content)

# Use cases for debate pattern
use_cases = [
    'Architecture decisions: microservices vs monolith',
    'Technology selection: PyTorch vs JAX for new project',
    'Business decisions: launch feature now vs wait for polish',
    'Research paper review: accept vs reject',
    'Code review: approve vs request changes',
]
print('Debate pattern use cases:')
for uc in use_cases:
    print(f'  • {uc}')
print()
print('Benefits over single-agent decision:')
print('  ✓ Surfaces hidden risks and edge cases')
print('  ✓ More balanced, less biased decision')
print('  ✓ Forces explicit reasoning on both sides')

## 4. Shared Memory and State Management

In [ ]:
from threading import RLock
from dataclasses import dataclass, field
from typing import Any
import time

@dataclass
class SharedWorkspace:
    """Thread-safe shared memory for multi-agent collaboration."""
    _data: dict[str, Any] = field(default_factory=dict)
    _lock: RLock = field(default_factory=RLock)
    _history: list[dict] = field(default_factory=list)
    
    def write(self, key: str, value: Any, agent: str):
        with self._lock:
            self._data[key] = value
            self._history.append({'time': time.time(), 'agent': agent, 'action': 'write', 'key': key})
    
    def read(self, key: str, default=None):
        with self._lock:
            return self._data.get(key, default)
    
    def append(self, key: str, item: Any, agent: str):
        with self._lock:
            if key not in self._data:
                self._data[key] = []
            self._data[key].append(item)
            self._history.append({'time': time.time(), 'agent': agent, 'action': 'append', 'key': key})
    
    def snapshot(self) -> dict:
        with self._lock:
            return dict(self._data)
    
    def audit_log(self) -> list[dict]:
        with self._lock:
            return list(self._history)

# Demo: agents collaborating on a research report
workspace = SharedWorkspace()

# Simulate agents writing to shared workspace
workspace.write('topic', 'LLM Inference Optimization', 'supervisor')
workspace.append('findings', 'Flash Attention reduces memory by 4x', 'researcher')
workspace.append('findings', 'PagedAttention enables 24x throughput', 'researcher')
workspace.write('analysis', 'Both techniques are complementary and can be combined', 'analyst')
workspace.write('draft_section', 'Introduction: LLM inference is bottlenecked by memory bandwidth...', 'writer')
workspace.write('critique', 'Introduction lacks specific numbers. Add: "memory BW: 3TB/s for A100"', 'critic')

print('Shared workspace state:')
for k, v in workspace.snapshot().items():
    if isinstance(v, list):
        print(f'  {k}: [{len(v)} items]')
        for item in v:
            print(f'    - {item}')
    else:
        print(f'  {k}: {str(v)[:60]}...' if len(str(v)) > 60 else f'  {k}: {v}')
print(f'\nAudit log: {len(workspace.audit_log())} operations by {len(set(h["agent"] for h in workspace.audit_log()))} agents')

## 5. Multi-Agent Research System

In [ ]:
import asyncio

RESEARCH_AGENTS = {
    'planner': """You are a research planner. Break down the research topic into 4-6 specific questions.
Each question should be independently researchable. Output as JSON list.""",
    
    'searcher': """You are a research specialist. Given a specific research question,
generate a comprehensive, well-sourced answer. Be specific and factual.""",
    
    'synthesizer': """You are a research synthesizer. Given multiple research findings,
create a coherent, well-structured report with clear sections.
Identify patterns, contradictions, and key insights across all findings.""",
    
    'fact_checker': """You are a fact checker. Review the research report for:
- Factual inaccuracies
- Unsupported claims
- Logical inconsistencies
Output a list of issues and corrected versions.""",
}

async def multi_agent_research(topic: str) -> dict:
    """Full research pipeline: plan → search in parallel → synthesize → fact-check."""
    workspace = SharedWorkspace()
    workspace.write('topic', topic, 'system')
    
    # Step 1: Planner creates research questions
    plan_response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'system', 'content': RESEARCH_AGENTS['planner']}, {'role': 'user', 'content': f'Topic: {topic}'}],
        response_format={'type': 'json_object'},
    )
    questions = json.loads(plan_response.choices[0].message.content).get('questions', [])
    workspace.write('research_questions', questions, 'planner')
    
    # Step 2: Searchers answer questions in parallel
    async def search_question(q: str) -> str:
        r = await async_client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'system', 'content': RESEARCH_AGENTS['searcher']}, {'role': 'user', 'content': q}],
        )
        return r.choices[0].message.content
    
    findings = await asyncio.gather(*[search_question(q) for q in questions])
    workspace.write('findings', dict(zip(questions, findings)), 'searchers')
    
    # Step 3: Synthesizer creates report
    findings_text = '\n\n'.join(f'Q: {q}\nA: {a}' for q, a in zip(questions, findings))
    report_r = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'system', 'content': RESEARCH_AGENTS['synthesizer']}, {'role': 'user', 'content': findings_text}],
    )
    report = report_r.choices[0].message.content
    workspace.write('draft_report', report, 'synthesizer')
    
    return workspace.snapshot()

print('Multi-agent research pipeline:')
print('  1. Planner:     1 agent  → breaks topic into questions')
print('  2. Searchers:   N agents → answer questions in PARALLEL')
print('  3. Synthesizer: 1 agent  → weaves answers into coherent report')
print('  4. Fact-checker: 1 agent → validates claims')
print()
print('Agents communicate through SharedWorkspace (not direct messaging)')
print('This decouples agents and provides full audit trail.')

## Exercises

1. **Dynamic team composition:** Build a supervisor that creates its worker team dynamically based on the task. For a coding task, it creates a coder + tester + reviewer; for a research task, it creates researchers + analyst + writer.

2. **Agent memory:** Implement persistent agent memory using a vector store. Each agent can `remember(key, value)` (stores embedding) and `recall(query)` (semantic search). Test how this improves multi-session task completion.

3. **Consensus mechanism:** Implement a voting mechanism where 5 agents independently answer a question, then vote on the best answer. Compare accuracy vs single-agent and majority-vote on a dataset of 50 factual questions.